# EV bidding on real auction logs (iPinYou)

The engine's demo A/B runs on simulated traffic. This notebook replays **real**
RTB logs instead: the iPinYou 2013 competition data, season 2 (5 advertisers,
6–12 June 2013). Each impression record carries the real market price (the
highest competing bid), the slot's floor price and whether it was clicked.

**Protocol** (as in Zhang et al., *Real-Time Bidding Benchmarking with iPinYou
Dataset*, 2014): replay the requests in time order; a bid wins when it beats the
logged market price and the floor; a win costs the market price and yields the
logged click; bidding stops when the budget is spent.

**Splits, all by time:** fit the CTR model on 6–9 June, tune each strategy's bid
level on 10 June, report on 11–12 June.

**Caveat:** logs exist only for auctions iPinYou won. A bid above iPinYou's
original one could have won auctions that were never logged. Every published
offline RTB evaluation on this data shares this limitation.

Add the dataset `lastsummer/ipinyou` as an input before running.

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
from pandas.api.types import union_categoricals
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# Find the dataset wherever Kaggle mounted it.
ROOT = next(r for r, dirs, _ in os.walk("/kaggle/input") if "training2nd" in dirs)
SEASON = os.path.join(ROOT, "training2nd")
OUT = "/kaggle/working"
print("dataset root:", ROOT)
print(open(os.path.join(ROOT, "known.data.bugs.txt"), errors="replace").read()[:3000])

# Column layout of the season 2/3 logs (Table 1 of the benchmark paper).
COLS = ["bidid", "timestamp", "logtype", "ipinyouid", "useragent", "IP", "region",
        "city", "adexchange", "domain", "url", "urlid", "slotid", "slotwidth",
        "slotheight", "slotvisibility", "slotformat", "slotprice", "creative",
        "bidprice", "payprice", "keypage", "advertiser", "usertag"]
FIT_DAYS, TUNE_DAYS, TEST_DAYS = ["06", "07", "08", "09"], ["10"], ["11", "12"]

In [ ]:
# Read one file's first rows raw, to check the layout before reading 12M rows.
first = sorted(glob.glob(os.path.join(SEASON, "imp.*.txt.bz2")))[0]
peek = pd.read_csv(first, sep="\t", header=None, nrows=3, quoting=3, dtype=str)
assert peek.shape[1] == len(COLS), f"expected {len(COLS)} columns, got {peek.shape[1]}"
peek.columns = COLS
peek.T

In [ ]:
def device_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    tablet = ua.str.contains("ipad|tablet") | (ua.str.contains("android") & ~ua.str.contains("mobile"))
    mobile = ua.str.contains("iphone|ipod|mobile|windows phone|symbian|blackberry")
    return pd.Series(np.select([tablet, mobile], ["tablet", "mobile"], "desktop"), index=ua.index)

def os_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("windows"), ua.str.contains("android"),
         ua.str.contains("iphone|ipad|ipod"), ua.str.contains("mac os"),
         ua.str.contains("linux")],
        ["windows", "android", "ios", "mac", "linux"], "other"), index=ua.index)

def browser_of(ua: pd.Series) -> pd.Series:
    ua = ua.str.lower().fillna("")
    return pd.Series(np.select(
        [ua.str.contains("msie|trident"), ua.str.contains("chrome"),
         ua.str.contains("firefox"), ua.str.contains("safari"), ua.str.contains("opera")],
        ["ie", "chrome", "firefox", "safari", "opera"], "other"), index=ua.index)

clicked = set()
for f in sorted(glob.glob(os.path.join(SEASON, "clk.*.txt.bz2"))):
    clicked.update(pd.read_csv(f, sep="\t", header=None, usecols=[0], quoting=3,
                               dtype=str)[0])
print(f"{len(clicked):,} clicked bid ids")

CATS = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
        "slotvisibility", "slotformat", "device", "os", "browser", "hour", "day"]
frames = []
for f in sorted(glob.glob(os.path.join(SEASON, "imp.*.txt.bz2"))):
    t0 = time.time()
    d = pd.read_csv(f, sep="\t", header=None, names=COLS, quoting=3, dtype=str,
                    usecols=["bidid", "timestamp", "useragent", "region", "city",
                             "adexchange", "domain", "slotwidth", "slotheight",
                             "slotvisibility", "slotformat", "slotprice",
                             "payprice", "advertiser"])
    out = pd.DataFrame({
        "ts": d["timestamp"].str[:14],                       # yyyymmddHHMMSS
        "click": d["bidid"].isin(clicked).astype("int8"),
        "floor": pd.to_numeric(d["slotprice"], errors="coerce").fillna(0).astype("int32"),
        "pay": pd.to_numeric(d["payprice"], errors="coerce").fillna(0).astype("int32"),
    })
    out["hour"] = d["timestamp"].str[8:10]
    out["day"] = d["timestamp"].str[6:8]
    out["slotsize"] = d["slotwidth"] + "x" + d["slotheight"]
    out["device"], out["os"], out["browser"] = (
        device_of(d["useragent"]), os_of(d["useragent"]), browser_of(d["useragent"]))
    for c in ["advertiser", "adexchange", "region", "city", "domain",
              "slotvisibility", "slotformat"]:
        out[c] = d[c].fillna("na")
    for c in CATS:
        out[c] = out[c].astype("category")
    frames.append(out)
    print(f"{os.path.basename(f)}: {len(out):,} rows, {out.click.sum()} clicks, {time.time() - t0:.0f}s")

df = pd.DataFrame({
    c: (union_categoricals([f[c] for f in frames]) if c in CATS
        else np.concatenate([f[c].to_numpy() for f in frames]))
    for c in frames[0].columns})
del frames
df = df.sort_values("ts", kind="stable").reset_index(drop=True)

# Sanity check against Table 3 of the paper (e.g. 1458: 3,083,056 imps, 2,454 clicks).
summary = df.groupby("advertiser", observed=True).agg(
    imps=("click", "size"), clicks=("click", "sum"), cost=("pay", "sum"))
summary["ctr"] = summary.clicks / summary.imps
summary["cpm"] = summary.cost / summary.imps
print(summary.to_string())
print(df.groupby("day", observed=True).size().to_string())

In [ ]:
# ── CTR model: logistic regression on bid-time features only ─────────────────
# Excluded on purpose: the market price (unknown until after the auction), the
# user and IP ids (memorisation, not generalisation) and the creative.
FEATURES = ["advertiser", "adexchange", "region", "city", "domain", "slotsize",
            "slotvisibility", "slotformat", "device", "os", "browser", "hour"]
fit = df[df.day.isin(FIT_DAYS)]
tune = df[df.day.isin(TUNE_DAYS)].reset_index(drop=True)
test = df[df.day.isin(TEST_DAYS)].reset_index(drop=True)
print(f"fit {len(fit):,} ({fit.click.sum()} clicks)  tune {len(tune):,} ({tune.click.sum()})  "
      f"test {len(test):,} ({test.click.sum()})")

ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, dtype=np.float32)
X_fit = ohe.fit_transform(fit[FEATURES].astype(str))
t0 = time.time()
model = LogisticRegression(C=0.1, max_iter=500)
model.fit(X_fit, fit.click)
print(f"fitted on {X_fit.shape[1]} one-hot features in {time.time() - t0:.0f}s")

def pctr(frame):
    return model.predict_proba(ohe.transform(frame[FEATURES].astype(str)))[:, 1]

tune["pctr"], test["pctr"] = pctr(tune), pctr(test)
metrics = {
    "auc": roc_auc_score(test.click, test.pctr),
    "logloss": log_loss(test.click, test.pctr),
    "brier": brier_score_loss(test.click, test.pctr),
    "mean_pctr": float(test.pctr.mean()), "test_ctr": float(test.click.mean()),
}
print({k: round(v, 5) for k, v in metrics.items()})
per_adv = {a: roc_auc_score(g.click, g.pctr) for a, g in test.groupby("advertiser", observed=True)
           if g.click.nunique() == 2}
print("AUC per advertiser:", {a: round(v, 3) for a, v in per_adv.items()})

# Calibration by pCTR decile: the EV bid multiplies pCTR by a value, so a model
# that is systematically high overbids. Clicks are rare, so deciles, not bins.
test["decile"] = pd.qcut(test.pctr.rank(method="first"), 10, labels=False)
print(test.groupby("decile").agg(mean_pctr=("pctr", "mean"), ctr=("click", "mean"),
                                 n=("click", "size")).to_string())

In [ ]:
# ── Replay ───────────────────────────────────────────────────────────────────
# The win rule has no separate floor check. The market price already includes
# the reserve (a winner never pays below the floor), and the paper notes the
# floor column was rescaled, so comparing a bid against it would mix units.
print("rows with market price below the logged floor:",
      f"{(test.pay < test.floor).mean():.2%}")

def replay(bids, pay, clicks, budget):
    """Requests in time order: win when the bid beats the market price, pay the
    market price, stop once the budget is spent."""
    win = bids > pay
    cost = np.where(win, pay, 0)
    won = win & (np.cumsum(cost) <= budget)
    return int(won.sum()), int(clicks[won].sum()), int(cost[won].sum())

def by_advertiser(frame):
    return {adv: {"pay": g["pay"].to_numpy(), "click": g["click"].to_numpy(),
                  "pctr": g["pctr"].to_numpy(), "block": (g["day"].astype(str)
                                                         + g["hour"].astype(str)).to_numpy()}
            for adv, g in frame.groupby("advertiser", observed=True)}

TUNE, TEST = by_advertiser(tune), by_advertiser(test)
# The EV bid's scale is fixed before the test period: the mean pCTR on the tune day.
NORM = {adv: a["pctr"].mean() for adv, a in TUNE.items()}

# flat: one bid for every request (the engine's flat CPM)
# ev:   bid = level * pCTR / mean pCTR, i.e. pCTR × a value per click, so the
#       level is the bid on an average request (the paper's linear bidder)
def bids_for(strategy, a, adv, level):
    if strategy == "flat":
        return np.full(len(a["pay"]), float(level))
    return level * a["pctr"] / NORM[adv]

STRATEGIES = ["flat", "ev"]
GRID = np.arange(2, 402, 2)
BUDGET_FRACTIONS = {"1/32": 1 / 32, "1/8": 1 / 8, "1/2": 1 / 2}

def run(a, adv, strategy, level, fraction, idx=None):
    if idx is not None:
        a = {k: v[idx] for k, v in a.items()}
    return replay(bids_for(strategy, a, adv, level), a["pay"], a["click"],
                  fraction * a["pay"].sum())

# Each strategy gets its own best bid level, chosen on the tune day: the most
# clicks for the budget, ties to the cheaper. Same grid, same rule, both arms.
levels = {}
for adv, a in TUNE.items():
    for s in STRATEGIES:
        for name, frac in BUDGET_FRACTIONS.items():
            scores = [(run(a, adv, s, b, frac), b) for b in GRID]
            levels[adv, s, name] = max(scores, key=lambda x: (x[0][1], -x[0][2]))[1]

rows = []
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            imps, clk, cost = run(a, adv, s, levels[adv, s, name], frac)
            rows.append(dict(advertiser=adv, budget=name, strategy=s,
                             level=int(levels[adv, s, name]), imps=imps, clicks=clk, cost=cost))
results = pd.DataFrame(rows)
results["ecpc"] = results.cost / results.clicks.replace(0, np.nan)
print(results.to_string(index=False))

In [ ]:
# ── Uncertainty: block bootstrap over test hours ─────────────────────────────
# Requests within an hour are correlated (same campaigns, same traffic), so
# whole hours are resampled, kept in time order, with the budget rescaled to the
# resample's own cost. Both strategies replay the SAME resample: the comparison
# is paired, and the tuned bid levels stay fixed.
rng = np.random.default_rng(0)
B = 300
BLOCKS = {adv: [np.flatnonzero(a["block"] == k) for k in np.unique(a["block"])]
          for adv, a in TEST.items()}   # np.unique sorts, so blocks are in time order

def pooled(sample):
    """Clicks and cost per (budget, strategy), summed over advertisers."""
    out = {}
    for adv, idx in sample.items():
        for name, frac in BUDGET_FRACTIONS.items():
            for s in STRATEGIES:
                _, clk, cost = run(TEST[adv], adv, s, levels[adv, s, name], frac, idx)
                out[name, s, "clicks"] = out.get((name, s, "clicks"), 0) + clk
                out[name, s, "cost"] = out.get((name, s, "cost"), 0) + cost
    return out

observed = pooled({adv: np.concatenate(b) for adv, b in BLOCKS.items()})
boot = []
for _ in range(B):
    boot.append(pooled({adv: np.concatenate([b[i] for i in np.sort(
        rng.integers(0, len(b), len(b)))]) for adv, b in BLOCKS.items()}))

def lift(o, name):         # extra clicks EV bidding won for the same budget
    return o[name, "ev", "clicks"] / o[name, "flat", "clicks"] - 1

def ecpc_change(o, name):  # change in cost per click
    return ((o[name, "ev", "cost"] / o[name, "ev", "clicks"])
            / (o[name, "flat", "cost"] / o[name, "flat", "clicks"]) - 1)

summary_rows = []
for name in BUDGET_FRACTIONS:
    lifts = np.array([lift(o, name) for o in boot])
    ecpcs = np.array([ecpc_change(o, name) for o in boot])
    summary_rows.append(dict(
        budget=name,
        flat_clicks=observed[name, "flat", "clicks"], ev_clicks=observed[name, "ev", "clicks"],
        click_lift=lift(observed, name),
        lift_ci=[float(np.quantile(lifts, 0.025)), float(np.quantile(lifts, 0.975))],
        ecpc_change=ecpc_change(observed, name),
        ecpc_ci=[float(np.quantile(ecpcs, 0.025)), float(np.quantile(ecpcs, 0.975))],
    ))
summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.float_format", "{:.3f}".format)
print(summary_df.to_string(index=False))

In [ ]:
# ── Robustness: is the gap an artefact of tuning on one noisy day? ───────────
# At a 1/32 budget the tune day holds only a handful of clicks per advertiser,
# so a strategy can draw an unlucky bid level. Here both strategies get their
# best possible level in hindsight, picked on the test days themselves. That
# flatters both equally: if EV still wins, the gain comes from the bidding rule,
# not from the flat bidder's level being badly chosen.
oracle = {}
for adv, a in TEST.items():
    for name, frac in BUDGET_FRACTIONS.items():
        for s in STRATEGIES:
            best = max((run(a, adv, s, b, frac) for b in GRID), key=lambda r: (r[1], -r[2]))
            oracle[name, s] = oracle.get((name, s), 0) + best[1]
for name in BUDGET_FRACTIONS:
    print(f"budget {name}: best flat {oracle[name, 'flat']} clicks, "
          f"best EV {oracle[name, 'ev']} clicks, "
          f"lift {oracle[name, 'ev'] / oracle[name, 'flat'] - 1:+.1%}")

In [ ]:
with open(f"{OUT}/ipinyou_replay_results.json", "w") as f:
    json.dump({
        "data": "iPinYou season 2 (training2nd), 5 advertisers, 6-12 June 2013",
        "splits": {"fit": FIT_DAYS, "tune": TUNE_DAYS, "test": TEST_DAYS},
        "rows": {"fit": int(len(fit)), "tune": int(len(tune)), "test": int(len(test))},
        "clicks": {"fit": int(fit.click.sum()), "tune": int(tune.click.sum()),
                   "test": int(test.click.sum())},
        "model": {"type": "logistic_regression", "C": 0.1, "features": FEATURES,
                  "metrics": metrics, "auc_per_advertiser": per_adv},
        "bootstrap": {"blocks": "test hours", "replicates": B},
        "summary": summary_df.to_dict("records"),
        "per_advertiser": results.to_dict("records"),
    }, f, indent=2, default=float)
print("wrote ipinyou_replay_results.json")